In [1]:
import os
import cv2
import numpy as np
from tqdm import tqdm
import tensorflow as tf
from scipy import ndimage
from tensorflow.keras import layers

2025-09-03 08:14:49.793805: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756887289.997594      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756887290.060893      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
# flag to check if model is loaded already or not
# it will prevent reloding model again and again
seg_model = None
refiner = None

In [3]:
# -------------------------
# Custom layers / utilities
# -------------------------
@tf.keras.utils.register_keras_serializable(package="lafr_custom")
class ResizeToTarget(layers.Layer):
    def __init__(self, method='bilinear', **kwargs):
        super().__init__(**kwargs)
        self.method = method
    def call(self, inputs):
        x, target = inputs
        target_shape = tf.shape(target)[1:3]
        return tf.image.resize(x, target_shape, method=self.method)
    def get_config(self):
        cfg = super().get_config(); cfg.update({"method": self.method}); return cfg

@tf.keras.utils.register_keras_serializable(package="Custom", name="ChannelAttention")
class ChannelAttention(layers.Layer):
    def __init__(self, reduction=16, **kwargs):
        super(ChannelAttention, self).__init__(**kwargs); self.reduction = reduction
    def build(self, input_shape):
        channel = int(input_shape[-1])
        self.shared_dense_one = layers.Dense(max(1, channel // self.reduction),
                                            activation='relu', kernel_initializer='he_normal', use_bias=True)
        self.shared_dense_two = layers.Dense(channel, kernel_initializer='he_normal', use_bias=True)
        super().build(input_shape)
    def call(self, inputs):
        avg_pool = tf.reduce_mean(inputs, axis=[1,2], keepdims=True)
        max_pool = tf.reduce_max(inputs, axis=[1,2], keepdims=True)
        avg_s = tf.squeeze(avg_pool, axis=[1,2]); max_s = tf.squeeze(max_pool, axis=[1,2])
        avg_out = self.shared_dense_two(self.shared_dense_one(avg_s))
        max_out = self.shared_dense_two(self.shared_dense_one(max_s))
        attn = tf.nn.sigmoid(avg_out + max_out)
        attn = tf.reshape(attn, (-1,1,1,int(attn.shape[-1])))
        return inputs * attn
    def get_config(self):
        cfg = super().get_config(); cfg.update({"reduction": self.reduction}); return cfg

@tf.keras.utils.register_keras_serializable(package="Custom", name="SpatialAttention")
class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super(SpatialAttention, self).__init__(**kwargs); self.kernel_size = kernel_size
    def build(self, input_shape):
        self.conv2d = layers.Conv2D(filters=1, kernel_size=self.kernel_size, padding='same',
                                    activation='sigmoid', kernel_initializer='he_normal')
        super().build(input_shape)
    def call(self, inputs):
        avg_pool = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(inputs, axis=-1, keepdims=True)
        concat = tf.concat([avg_pool, max_pool], axis=-1)
        attention = self.conv2d(concat)
        return inputs * attention
    def get_config(self):
        cfg = super().get_config(); cfg.update({"kernel_size": self.kernel_size}); return cfg

In [4]:
# ---------------------------
# Postprocessing helper
# ---------------------------
def postprocess_mask(mask_prob, threshold=0.5, min_size=100):
    mask = (mask_prob >= threshold).astype(np.uint8)
    labeled, n = ndimage.label(mask)
    if n == 0:
        return np.zeros_like(mask, dtype=np.uint8)
    counts = np.bincount(labeled.ravel()); counts[0] = 0
    largest = counts.argmax()
    out = (labeled == largest).astype(np.uint8)
    if out.sum() < min_size:
        return np.zeros_like(out, dtype=np.uint8)
    return out

In [5]:
# ---------------------------
# TTA + refiner prediction
# ---------------------------
def predict_with_tta_and_refiner(model, refiner_model, image_rgb_norm,
                                input_size=(256,256),
                                tta_transforms=('none','flip_lr','flip_ud'),
                                threshold=0.5, min_size=100):
    H, W = image_rgb_norm.shape[:2]
    inp_w, inp_h = input_size
    inp = cv2.resize((image_rgb_norm * 255.0).astype(np.uint8), (inp_w, inp_h)).astype(np.float32) / 255.0

    batch = []
    for t in tta_transforms:
        if t == 'none': batch.append(inp)
        elif t == 'flip_lr': batch.append(np.fliplr(inp))
        elif t == 'flip_ud': batch.append(np.flipud(inp))
        else: batch.append(inp)
    batch = np.stack(batch).astype(np.float32)

    preds = model.predict(batch, verbose=0)

    corrected = []
    for p, t in zip(preds, tta_transforms):
        if t == 'none': corrected.append(p)
        elif t == 'flip_lr': corrected.append(np.fliplr(p))
        elif t == 'flip_ud': corrected.append(np.flipud(p))
        else: corrected.append(p)
    avg = np.mean(np.stack(corrected), axis=0)

    if avg.ndim == 3:
        avg_chan = avg[...,0]
    else:
        avg_chan = np.squeeze(avg)

    if refiner_model is not None:
        ref_in_img = inp
        ref_in_avg = np.expand_dims(avg_chan, axis=-1)
        ref_in = np.concatenate([ref_in_img, ref_in_avg], axis=-1)[None,...].astype(np.float32)
        refined = refiner_model.predict(ref_in, verbose=0)[0,...,0]
        refined_map = refined
    else:
        refined_map = avg_chan

    prob_map_resized = cv2.resize(refined_map.astype(np.float32), (W, H))
    processed = postprocess_mask(prob_map_resized, threshold=threshold, min_size=min_size)
    return processed.astype(np.uint8), prob_map_resized.astype(np.float32)


In [6]:
# ---------------------------
# Single-image processing + saving (mask path passed in)
# ---------------------------
def process_and_save_image_with_maskpath(src_path, seg_model, refiner_model,
                                        input_size, threshold, min_size,
                                        out_img_path, out_mask_path=None, out_prob_path=None):
    bgr = cv2.imread(src_path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise RuntimeError(f"Could not read image: {src_path}")
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    rgb_norm = rgb.astype(np.float32) / 255.0

    bin_mask, prob_map = predict_with_tta_and_refiner(seg_model, refiner_model, rgb_norm,
                                                      input_size=input_size,
                                                      tta_transforms=('none','flip_lr','flip_ud'),
                                                      threshold=threshold, min_size=min_size)
    # Apply mask (background black)
    mask_3ch = np.stack([bin_mask]*3, axis=-1)
    masked_rgb = (rgb * mask_3ch).astype(np.uint8)

    os.makedirs(os.path.dirname(out_img_path), exist_ok=True)
    cv2.imwrite(out_img_path, cv2.cvtColor(masked_rgb, cv2.COLOR_RGB2BGR))

    if out_mask_path:
        os.makedirs(os.path.dirname(out_mask_path), exist_ok=True)
        cv2.imwrite(out_mask_path, (bin_mask * 255).astype(np.uint8))
    if out_prob_path:
        os.makedirs(os.path.dirname(out_prob_path), exist_ok=True)
        p_uint8 = np.clip((prob_map * 255.0), 0, 255).astype(np.uint8)
        cv2.imwrite(out_prob_path, p_uint8)

In [7]:
# ---------------------------
# Helper: compute mask-folder relative path for a given image rel path
# ---------------------------
def _compute_mask_dir_rel(rel, input_dir):
    """
    Given rel path of file (relative to input_dir), compute a relative path
    where the mask should go. Rules:
      - If file is in a folder (rel has parent), append '_mask' to the last parent folder:
          rel = 'a/b/c/img.jpg' -> mask_dir_rel = 'a/b/c_mask'
      - If file is directly under input_dir (rel has no parent), use input_dir basename + '_mask':
          rel = 'img.jpg' and input_dir='/path/to/VASC' -> mask_dir_rel = 'VASC_mask'
    """
    parent_rel = os.path.dirname(rel)  # '' if file at root of input_dir
    if parent_rel == '':
        base_input = os.path.basename(input_dir.rstrip("/\\"))
        return base_input + "_mask"
    parent_prefix = os.path.dirname(parent_rel)  # may be ''
    last = os.path.basename(parent_rel)
    if parent_prefix == '':
        return last + "_mask"
    else:
        return os.path.join(parent_prefix, last + "_mask")

In [8]:
# ---------------------------
# Main pipeline (masks in foldername_mask)
# ---------------------------
def run_pipeline(seg_model_path, refiner_model_path, input_dir, output_dir,
                 input_size=(256,256), threshold=0.5, min_size=100,
                 exts=('.jpg','.jpeg','.png','.tif','.tiff'),
                 save_masks=True, save_probs=False, preserve_structure=True):
    """
    - seg_model_path: required (.h5)
    - refiner_model_path: optional (.h5) or None
    - input_dir: folder with images (recursively searched)
    - output_dir: root for masked images (preserve structure by default)
    Masks will be saved into sibling folders named '<foldername>_mask' as explained above.
    """
    global seg_model, refiner   # to access those model globally
    
    custom_objects = {
        "ChannelAttention": ChannelAttention,
        "SpatialAttention": SpatialAttention,
        "ResizeToTarget": ResizeToTarget,
    }

    # Load segmentation model only if not already loaded
    if seg_model is None:
        print("Loading segmentation model:", seg_model_path)
        seg_model = tf.keras.models.load_model(seg_model_path, compile=False, custom_objects=custom_objects)
        print("Segmentation model loaded.")
    else:
        print("Segmentation model already in memory, skipping reload.")

    # Load refiner model only if path is provided and not already loaded
    if refiner_model_path:
        if refiner is None:
            if os.path.exists(refiner_model_path):
                print("Loading refiner model:", refiner_model_path)
                refiner = tf.keras.models.load_model(refiner_model_path, compile=False, custom_objects=custom_objects)
                print("Refiner model loaded.")
            else:
                print("Refiner path provided but not found. Continuing without refiner:", refiner_model_path)
        else:
            print("Refiner model already in memory, skipping reload.")

    # collect image files recursively
    files = []
    for root, _, filenames in os.walk(input_dir):
        for fn in filenames:
            if os.path.splitext(fn)[1].lower() in exts:
                files.append(os.path.join(root, fn))
    print(f"Found {len(files)} image(s) under {input_dir}")

    processed = []
    # progress bar (plain tqdm works without ipywidgets)
    for src in tqdm(files, desc="Processing images"):
        rel = os.path.relpath(src, input_dir)  # e.g., 'lesion123/img1.jpg' or 'img1.jpg'
        # --- compute output masked image path (preserve structure or flatten) ---
        if preserve_structure:
            out_img_path = os.path.join(output_dir, rel)
        else:
            base = os.path.basename(src)
            out_img_path = os.path.join(output_dir, base)
            if os.path.exists(out_img_path):
                name, ext = os.path.splitext(base); i = 1
                while os.path.exists(out_img_path):
                    out_img_path = os.path.join(output_dir, f"{name}_{i}{ext}"); i += 1

        # --- compute where to save mask ---
        if save_masks:
            if preserve_structure:
                # mask folder is same relative parent but with last folder name appended with _mask
                mask_dir_rel = _compute_mask_dir_rel(rel, input_dir)   # e.g., 'lesion123_mask' or 'a/b/c_mask'
                mask_fname = os.path.splitext(os.path.basename(rel))[0] + "_mask.png"
                out_mask_path = os.path.join(output_dir, mask_dir_rel, mask_fname)
            else:
                # flattened -> a single mask folder next to output_dir: output_dir + "_mask"
                mask_root = output_dir.rstrip("/\\") + "_mask"
                os.makedirs(mask_root, exist_ok=True)
                mask_fname = os.path.splitext(os.path.basename(rel))[0] + "_mask.png"
                # avoid name collisions in flattened mask folder
                out_mask_path = os.path.join(mask_root, mask_fname)
                if os.path.exists(out_mask_path):
                    name = os.path.splitext(mask_fname)[0]; ext = os.path.splitext(mask_fname)[1]; i = 1
                    while os.path.exists(out_mask_path):
                        out_mask_path = os.path.join(mask_root, f"{name}_{i}{ext}"); i += 1
        else:
            out_mask_path = None

        # --- compute optional prob path (saved next to mask) ---
        if save_probs:
            if out_mask_path is not None:
                out_prob_path = os.path.splitext(out_mask_path)[0] + "_prob.png"
            else:
                # if mask disabled but probs requested, put prob alongside masked image (fallback)
                out_prob_path = os.path.splitext(out_img_path)[0] + "_prob.png"
        else:
            out_prob_path = None

        try:
            process_and_save_image_with_maskpath(src, seg_model, refiner,
                                                input_size=input_size, threshold=threshold, min_size=min_size,
                                                out_img_path=out_img_path,
                                                out_mask_path=out_mask_path,
                                                out_prob_path=out_prob_path)
            processed.append((out_img_path, out_mask_path))
        except Exception as e:
            print(f"[error] {src} -> {e}")

    print(f"Done. Processed {len(processed)}/{len(files)} images. Output dir: {output_dir}")
    return processed

In [9]:
# -------------------------
# Wrapper: run pipeline on all immediate subfolders of a parent folder
# -------------------------
def run_on_subfolders(parent_input_dir,
                      output_root_dir,
                      seg_model_path,
                      refiner_model_path=None,
                      input_size=(256,256),
                      threshold=0.5,
                      min_size=100,
                      save_masks=True,
                      save_probs=False,
                      preserve_structure=True):
    """
    For every immediate subfolder under `parent_input_dir`, call run_pipeline(...) on that subfolder.
    Output for subfolder 'parent_input_dir/<sub>' will be placed into 'output_root_dir/<sub>' (preserve top-level names).
    If there are images directly under parent_input_dir (not inside a subfolder), they will be processed into
    output_root_dir/<parent_basename>/...
    Returns a dict mapping subfolder name -> list of processed (image, mask) pairs (the same return format as run_pipeline).
    """
    parent_input_dir = os.path.abspath(parent_input_dir)
    output_root_dir = os.path.abspath(output_root_dir)
    os.makedirs(output_root_dir, exist_ok=True)

    items = sorted(os.listdir(parent_input_dir))
    subdirs = [d for d in items if os.path.isdir(os.path.join(parent_input_dir, d))]

    results = {}

    # 1) Process each immediate subfolder separately
    for sub in subdirs:
        in_dir = os.path.join(parent_input_dir, sub)
        out_dir = os.path.join(output_root_dir, sub)  # preserve top-level folder name
        os.makedirs(out_dir, exist_ok=True)
        print(f"\n>>> Processing folder: {in_dir}  -->  {out_dir}")
        processed = run_pipeline(
            seg_model_path=seg_model_path,
            refiner_model_path=refiner_model_path,
            input_dir=in_dir,
            output_dir=out_dir,
            input_size=input_size,
            threshold=threshold,
            min_size=min_size,
            save_masks=save_masks,
            save_probs=save_probs,
            preserve_structure=preserve_structure
        )
        results[sub] = processed

    # 2) If there are any images directly in parent_input_dir, process them too
    root_images = [f for f in items if os.path.isfile(os.path.join(parent_input_dir, f)) and os.path.splitext(f)[1].lower() in ('.jpg','.jpeg','.png','.tif','.tiff')]
    if root_images:
        parent_basename = os.path.basename(parent_input_dir.rstrip("/\\"))
        in_dir = parent_input_dir
        out_dir = os.path.join(output_root_dir, parent_basename)
        os.makedirs(out_dir, exist_ok=True)
        print(f"\n>>> Processing images in parent folder root: {in_dir}  -->  {out_dir}")
        processed_root = run_pipeline(
            seg_model_path=seg_model_path,
            refiner_model_path=refiner_model_path,
            input_dir=in_dir,
            output_dir=out_dir,
            input_size=input_size,
            threshold=threshold,
            min_size=min_size,
            save_masks=save_masks,
            save_probs=save_probs,
            preserve_structure=preserve_structure
        )
        results[parent_basename] = processed_root

    if not subdirs and not root_images:
        print(f"No subfolders or images found in {parent_input_dir}")

    return results

In [10]:
# -------------------------
# Model Configuration
# -------------------------

SEG_MODEL_PATH = r"/kaggle/input/final-segmentaion-model/lafr_outputs/segmentation_best.h5"
REFINER_MODEL_PATH = r"/kaggle/input/final-segmentaion-model/lafr_outputs/refiner_best.h5"  
INPUT_SIZE = (256, 256)   # (W, H)
THRESHOLD = 0.5
MIN_SIZE = 100
SAVE_MASKS = True
SAVE_PROBS = False
PRESERVE_STRUCTURE = True

In [11]:
# -------------------------
# usage for Ham10000 Dataset
# -------------------------
PARENT_FOLDER = r"/kaggle/input/ham10000-256-resized-without-aug/HAM10000_organized"            
OUTPUT_ROOT = r"HAM10000_organized_masked_out"   

results_dict = run_on_subfolders(
    parent_input_dir=PARENT_FOLDER,
    output_root_dir=OUTPUT_ROOT,
    seg_model_path=SEG_MODEL_PATH,
    refiner_model_path=REFINER_MODEL_PATH,
    input_size=INPUT_SIZE,
    threshold=THRESHOLD,
    min_size=MIN_SIZE,
    save_masks=SAVE_MASKS,
    save_probs=SAVE_PROBS,
    preserve_structure=PRESERVE_STRUCTURE
)


>>> Processing folder: /kaggle/input/ham10000-256-resized-without-aug/HAM10000_organized/akiec  -->  /kaggle/working/HAM10000_organized_masked_out/akiec
Loading segmentation model: /kaggle/input/final-segmentaion-model/lafr_outputs/segmentation_best.h5


I0000 00:00:1756887303.970731      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Segmentation model loaded.
Loading refiner model: /kaggle/input/final-segmentaion-model/lafr_outputs/refiner_best.h5
Refiner model loaded.
Found 327 image(s) under /kaggle/input/ham10000-256-resized-without-aug/HAM10000_organized/akiec


Processing images:   0%|          | 0/327 [00:00<?, ?it/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1756887311.617499      60 service.cc:148] XLA service 0x7faec802fd80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1756887311.618248      60 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1756887312.166623      60 cuda_dnn.cc:529] Loaded cuDNN version 90300
E0000 00:00:1756887314.488611      60 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1756887314.721023      60 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1756887315.177619      60 gpu_timer.cc:82] Delay kernel timed out: m

Done. Processed 327/327 images. Output dir: /kaggle/working/HAM10000_organized_masked_out/akiec

>>> Processing folder: /kaggle/input/ham10000-256-resized-without-aug/HAM10000_organized/bcc  -->  /kaggle/working/HAM10000_organized_masked_out/bcc
Segmentation model already in memory, skipping reload.
Refiner model already in memory, skipping reload.
Found 514 image(s) under /kaggle/input/ham10000-256-resized-without-aug/HAM10000_organized/bcc


...

Done. Processed 1113/1113 images. Output dir: /kaggle/working/HAM10000_organized_masked_out/mel

>>> Processing folder: /kaggle/input/ham10000-256-resized-without-aug/HAM10000_organized/nv  -->  /kaggle/working/HAM10000_organized_masked_out/nv
Segmentation model already in memory, skipping reload.
Refiner model already in memory, skipping reload.
Found 6705 image(s) under /kaggle/input/ham10000-256-resized-without-aug/HAM10000_organized/nv


Processing images: 100%|██████████| 6705/6705 [22:35<00:00,  4.95it/s]


Done. Processed 6705/6705 images. Output dir: /kaggle/working/HAM10000_organized_masked_out/nv

>>> Processing folder: /kaggle/input/ham10000-256-resized-without-aug/HAM10000_organized/vasc  -->  /kaggle/working/HAM10000_organized_masked_out/vasc
Segmentation model already in memory, skipping reload.
Refiner model already in memory, skipping reload.
Found 142 image(s) under /kaggle/input/ham10000-256-resized-without-aug/HAM10000_organized/vasc


Processing images: 100%|██████████| 142/142 [00:26<00:00,  5.46it/s]


Done. Processed 142/142 images. Output dir: /kaggle/working/HAM10000_organized_masked_out/vasc


In [12]:
# -------------------------
# usage for MILK10K Dataset
# -------------------------
PARENT_FOLDER = r"/kaggle/input/milk10k-256resized-witout-aug/MILK10k_Organized"            
OUTPUT_ROOT = r"MILK10k_Organized_masked_out"   

results_dict = run_on_subfolders(
    parent_input_dir=PARENT_FOLDER,
    output_root_dir=OUTPUT_ROOT,
    seg_model_path=SEG_MODEL_PATH,
    refiner_model_path=REFINER_MODEL_PATH,
    input_size=INPUT_SIZE,
    threshold=THRESHOLD,
    min_size=MIN_SIZE,
    save_masks=SAVE_MASKS,
    save_probs=SAVE_PROBS,
    preserve_structure=PRESERVE_STRUCTURE
)


>>> Processing folder: /kaggle/input/milk10k-256resized-witout-aug/MILK10k_Organized/AKIEC  -->  /kaggle/working/MILK10k_Organized_masked_out/AKIEC
Segmentation model already in memory, skipping reload.
Refiner model already in memory, skipping reload.
Found 606 image(s) under /kaggle/input/milk10k-256resized-witout-aug/MILK10k_Organized/AKIEC


Processing images: 100%|██████████| 606/606 [01:55<00:00,  5.24it/s]


Done. Processed 606/606 images. Output dir: /kaggle/working/MILK10k_Organized_masked_out/AKIEC

>>> Processing folder: /kaggle/input/milk10k-256resized-witout-aug/MILK10k_Organized/BCC  -->  /kaggle/working/MILK10k_Organized_masked_out/BCC
Segmentation model already in memory, skipping reload.
Refiner model already in memory, skipping reload.
Found 5044 image(s) under /kaggle/input/milk10k-256resized-witout-aug/MILK10k_Organized/BCC


Processing images: 100%|██████████| 5044/5044 [16:37<00:00,  5.06it/s]


Done. Processed 5044/5044 images. Output dir: /kaggle/working/MILK10k_Organized_masked_out/BCC

>>> Processing folder: /kaggle/input/milk10k-256resized-witout-aug/MILK10k_Organized/BEN_OTH  -->  /kaggle/working/MILK10k_Organized_masked_out/BEN_OTH
Segmentation model already in memory, skipping reload.
Refiner model already in memory, skipping reload.
Found 88 image(s) under /kaggle/input/milk10k-256resized-witout-aug/MILK10k_Organized/BEN_OTH


...

Done. Processed 1492/1492 images. Output dir: /kaggle/working/MILK10k_Organized_masked_out/NV

>>> Processing folder: /kaggle/input/milk10k-256resized-witout-aug/MILK10k_Organized/SCCKA  -->  /kaggle/working/MILK10k_Organized_masked_out/SCCKA
Segmentation model already in memory, skipping reload.
Refiner model already in memory, skipping reload.
Found 946 image(s) under /kaggle/input/milk10k-256resized-witout-aug/MILK10k_Organized/SCCKA


Processing images: 100%|██████████| 946/946 [03:05<00:00,  5.09it/s]


Done. Processed 946/946 images. Output dir: /kaggle/working/MILK10k_Organized_masked_out/SCCKA

>>> Processing folder: /kaggle/input/milk10k-256resized-witout-aug/MILK10k_Organized/VASC  -->  /kaggle/working/MILK10k_Organized_masked_out/VASC
Segmentation model already in memory, skipping reload.
Refiner model already in memory, skipping reload.
Found 94 image(s) under /kaggle/input/milk10k-256resized-witout-aug/MILK10k_Organized/VASC


Processing images: 100%|██████████| 94/94 [00:19<00:00,  4.89it/s]


Done. Processed 94/94 images. Output dir: /kaggle/working/MILK10k_Organized_masked_out/VASC
